# 🔀 使用 Azure AI Foundry 的条件代理工作流（Python）## 📋 高级基于决策的工作流教程本笔记本演示了使用 Azure AI Foundry 和 Microsoft Agent Framework 的**条件工作流模式**。您将学习如何构建智能、基于决策的工作流，这些工作流根据内容分析、业务规则和 AI 驱动的决策制定来动态路由处理。## 🎯 学习目标### 🧠 **智能决策制定**- **条件逻辑**：基于 AI 分析和业务规则实现动态分支- **内容感知路由**：基于内容分析和分类路由工作流路径- **自适应处理**：根据实时条件和数据调整工作流行为- **Azure AI 集成**：利用 Azure AI Foundry 的高级功能进行决策制定### 🔀 **高级工作流模式**- **决策树**：构建具有多个分支点的复杂决策结构- **基于规则的处理**：实现业务逻辑和合规要求- **动态工作流修改**：根据运行时条件调整工作流- **上下文感知操作**：基于累积的工作流上下文做出决策### 🏢 **企业条件应用**- **文档分类**：将文档路由到适当的处理工作流- **客户服务分类**：自动将查询路由到专门的处理工作流- **合规处理**：基于内容类型和法规应用不同的验证规则- **质量保证**：基于质量指标将内容路由通过不同的审核流程## ⚙️ 先决条件和设置### 📦 **安装和依赖项**此工作流需要特定的 Azure AI Foundry 集成安装步骤：```bashpip install agent-framework-azure-ai -U ```### 🔑 **Azure AI Foundry 配置****所需的 Azure 资源：**- 部署了适当模型的 Azure AI Foundry 工作区- 具有必要权限的 Azure 订阅- 配置的 Azure CLI 身份验证**身份验证设置：**```bash# Azure CLI 身份验证az loginaz account set --subscription "your-subscription-id"azd auth login```### 🏗️ **条件工作流架构**```mermaidgraph TD    A[输入文档/请求] --> B[初始分析代理]    B --> C{决策点}    C -->|条件 1| D[工作流路径 A]    C -->|条件 2| E[工作流路径 B]    C -->|条件 3| F[工作流路径 C]    D --> G[专门处理 A]    E --> H[专门处理 B]    F --> I[专门处理 C]    G --> J[结果集成]    H --> J    I --> J    J --> K[最终输出]```**关键组件：**- **分析代理**：评估内容并做出路由决策的 AI 代理- **决策点**：确定工作流路径的条件逻辑- **专门处理器**：针对特定内容类型或场景优化的不同代理- **集成层**：组合来自不同工作流路径的结果## 🎨 **条件工作流设计模式**### 📋 **文档处理分类**```文档输入 → 内容分析 → 分类 → 专门处理工作流```### 🎯 **客户服务路由**```客户查询 → 意图分析 → 紧急程度评估 → 路由到专家团队```### 🔍 **质量保证工作流**```内容输入 → 质量指标 → 风险评估 → 适当的审核流程```### 📊 **商业智能管道**```数据输入 → 源分析 → 处理规则 → 专门的分析工作流```## 🏢 **企业优势**### 🎯 **智能自动化**- **智能路由**：自动将工作引导至最合适的处理路径- **自适应行为**：基于模式和结果学习并调整的工作流- **业务规则集成**：纳入复杂的业务逻辑和合规要求- **上下文感知处理**：基于完整的工作流上下文和历史做出决策### 📈 **运营效率**- **减少人工干预**：自动化决策制定减少了人工路由的需要- **专门处理**：每个工作流路径针对特定场景进行了优化- **资源优化**：根据内容类型高效分配处理资源- **更快的解决时间**：直接路由到适当的专家和流程### 🛡️ **治理和控制**- **审计跟踪**：完整记录决策点和路由理由- **合规执行**：自动应用监管和政策要求- **风险管理**：通过增强的安全和审核流程路由高风险内容- **质量保证**：基于内容特征确保适当级别的审核### 📊 **分析和优化**- **决策分析**：跟踪路由决策和工作流路径的有效性- **性能指标**：衡量不同工作流分支的效率- **持续改进**：识别条件逻辑中的优化机会- **商业智能**：深入了解内容模式和处理需求让我们构建智能、基于决策的 AI 工作流！🚀

In [ ]:
! pip install agent-framework-azure-ai -U 

requirements.txt 和 constraints.txt - 在 ./Installation 目录中请将 .env.examples 复制为 .env

**注意** 选择 gpt-4.1-mini

In [ ]:
import osfrom dataclasses import dataclassfrom typing_extensions import Literalfrom pydantic import BaseModel

In [ ]:
from azure.identity.aio import AzureCliCredentialfrom dotenv import load_dotenvfrom agent_framework import HostedWebSearchToolfrom agent_framework.azure import AzureAIAgentClientfrom agent_framework import (    AgentExecutor,    AgentExecutorRequest    AgentExecutorResponse    HostedCodeInterpreterTool    ChatMessage    Role    WorkflowBuilder    WorkflowContext    WorkflowEvent    executor    WorkflowViz)from azure.ai.agents.models import BingGroundingTool,CodeInterpreterTool

In [ ]:
load_dotenv()

In [ ]:
EvangelistInstructions = """你是一位技术布道师，需要为技术教程创建初稿。1. 大纲中的每个知识点都必须包含链接。按照链接访问与大纲中知识点相关的内容。对该内容进行扩展。2. 每个知识点都必须详细解释。3. 根据条目要求重写内容，包括标题、大纲和相应内容。无需完全按照大纲顺序。4. 内容必须超过200字。4. 以Markdown格式输出草稿。将'draft_content'设置为草稿内容。5. 以JSON格式返回结果，包含字段'draft_content'（字符串）。"""ContentReviewerInstructions = """你是一家出版公司的内容审阅者。你需要检查教程的草稿内容是否符合以下要求：1. 如果草稿内容少于200字，将'review_result'设置为'No'，'reason'设置为'Content is too short'。如果草稿内容超过200字，将'review_result'设置为'Yes'，'reason'设置为'The content is good'。2. 将'draft_content'设置为原始草稿内容。3. 以JSON格式返回结果，包含字段'review_result'（Yes或No之一）、'reason'（字符串）和'draft_content'（字符串）。"""PublisherInstructions = """你是内容发布者，运行代码将教程的草稿内容保存为Markdown文件。保存文件的名称标有当前日期和时间，例如年月日时分秒。注意，如果是1-9，需要添加0，例如  20240101123045.md。 """

In [ ]:
OUTLINE_Content ="""# 介绍 AI Agent## 什么是 AI Agenthttps://github.com/microsoft/ai-agents-for-beginners/tree/main/01-intro-to-ai-agents***注意*** 不要创建任何示例代码 ## 介绍 Azure AI Foundry Agent Service https://learn.microsoft.com/en-us/azure/ai-foundry/agents/overview***注意*** 不要创建任何示例代码 ## Microsoft Agent Framework https://github.com/microsoft/agent-framework/tree/main/docs/docs-templates***注意*** 不要创建任何示例代码 """

In [ ]:
conn_id = os.environ["BING_CONNECTION_ID"]  # 确保设置了 BING_CONNECTION_NAME 环境变量# 初始化 Bing Grounding 工具bing = BingGroundingTool(connection_id=conn_id)code_interpreter = CodeInterpreterTool()

In [ ]:
class EvangelistAgent(BaseModel):    draft_content: strclass ReviewAgent(BaseModel):    review_result: Literal["Yes", "No"]    reason: str    draft_content: strclass PublisherAgent(BaseModel):    file_path: str@dataclassclass ReviewResult:    review_result: str    reason: str    draft_content: str@executor(id="to_reviewer_result")async def to_reviewer_result(response: AgentExecutorResponse, ctx: WorkflowContext[ReviewResult]) -> None:    print(f"从审阅者代理收到的原始响应: {response.agent_run_response.text}")    parsed = ReviewAgent.model_validate_json(response.agent_run_response.text)    await ctx.send_message(        ReviewResult(            review_result=parsed.review_result,            reason=parsed.reason,            draft_content=parsed.draft_content,        )    )def select_targets(review: ReviewResult, target_ids: list[str]) -> list[str]:        # 顺序: [handle_review, submit_to_email_assistant, summarize_email, handle_uncertain]        handle_review_id, save_draft_id = target_ids        if review.review_result == "Yes":            return [save_draft_id]        else:            return [handle_review_id]        @executor(id="handle_review")async def handle_review(review: ReviewResult, ctx: WorkflowContext[str]) -> None:    if review.review_result == "No":        await ctx.yield_output(f"审阅失败: {review.reason}, 请修改草稿。")    else:        await ctx.send_message(            AgentExecutorRequest(messages=[ChatMessage(Role.USER, text=review.draft_content)], should_respond=True)        )@executor(id="save_draft")async def save_draft(review: ReviewResult, ctx: WorkflowContext[AgentExecutorRequest]) -> None:    # 仅由 selection_func 为长的 NotSpam 电子邮件调用    await ctx.send_message(        AgentExecutorRequest(messages=[ChatMessage(Role.USER, text=review.draft_content)], should_respond=True)    )

In [ ]:
from IPython.display import SVG, display, HTML

In [ ]:
class DatabaseEvent(WorkflowEvent): ...

In [ ]:
async with (        AzureCliCredential() as credential,        AzureAIAgentClient(async_credential=credential) as chat_client,    ):          try:                evangelist_agent = AgentExecutor(chat_client.create_agent(                    instructions= (EvangelistInstructions),                    tools=[HostedWebSearchTool()],                    # response_format=EvangelistAgent                ),  id="evangelist_agent")                reviewer_agent = AgentExecutor(chat_client.create_agent(                    instructions=(ContentReviewerInstructions),                    # response_format=ReviewAgent                ), id="reviewer_agent")                publisher_agent = AgentExecutor(chat_client.create_agent(                    instructions=PublisherInstructions                    tools=HostedCodeInterpreterTool()                    response_format=PublisherAgent                ), id="publisher_agent")                workflow = (                    WorkflowBuilder()                        .set_start_executor(evangelist_agent)                        .add_edge(evangelist_agent, reviewer_agent)                        .add_edge(reviewer_agent, to_reviewer_result)                        .add_multi_selection_edge_group(                            to_reviewer_result                            [handle_review, save_draft]                            selection_func=select_targets                        )                        .add_edge(save_draft, publisher_agent)                        .build()                )                # workflow = SequentialBuilder().participants([evangelist_chat_agent, reviewer_chat_agent, publisher_chat_agent]).build()                print("生成工作流可视化...")                viz = WorkflowViz(workflow)                # 打印出 mermaid 字符串。                print("Mermaid 字符串: ")                print("======")                print(viz.to_mermaid())                print("=======")                # 打印出 DiGraph 字符串。                print("DiGraph 字符串: ")                print("=======")                print(viz.to_digraph())                print("=======")                svg_file = viz.export(format="svg")                print(f"SVG 文件保存到: {svg_file}")                if svg_file and os.path.exists(svg_file):                    try:                        # 首选: 直接 SVG 渲染                        display(SVG(filename=svg_file))                    except Exception as e:                        print(f"⚠️ 直接 SVG 渲染失败: {e}。回退到原始 HTML。")                        try:                            with open(svg_file, "r", encoding="utf-8") as f:                                svg_text = f.read()                            display(HTML(svg_text))                        except Exception as inner:                            print(f"❌ 回退 HTML 渲染也失败: {inner}")                else:                    print("❌ 未找到 SVG 文件。确保 viz.export(format='svg') 成功运行。")                                task = """                    你是一位布道师，需要根据以下大纲和大纲对应链接中提供的内容撰写草稿。草稿创建后，审阅者会检查它是否符合要求，如果符合要求，将提交给发布者并保存为 Markdown 文件，否则需要重写草稿直到符合要求。                        提供的大纲内容和相关链接如下：                    """ + OUTLINE_Content                                async for event in workflow.run_stream(task):                    if isinstance(event, DatabaseEvent):                        print(f"{event}")                    if isinstance(event, WorkflowEvent):                        print(f"工作流输出: {event.data}")        finally:            print("完成")